In [ ]:
# ============================================================================
# EVENT STUDY: Match Results vs Dortmund Stock Price
# 3 return types (Simple, Log, Market-Adjusted Abnormal) x
# 3 event windows (Day 0, Day 0+1, Day 0+2) = 9 combinations
# ============================================================================
import pandas as pd
import numpy as np
from scipy import stats

# --- Load Dortmund price data ---
stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"]).dt.date
stock_dortmund = stock_dortmund[["Date", "Price"]].sort_values("Date").reset_index(drop=True)

# --- Load match data ---
match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")
match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"]).dt.date

dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append('Win')
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append('Draw')
    else:
        dortmund_result.append('Loss')
match_dortmund['Result'] = dortmund_result
match_dortmund = match_dortmund[['Date', 'Result']].sort_values('Date').reset_index(drop=True)

# --- Load DAX market index data ---
dax = pd.read_csv("DAX Historical Data.csv")
dax["Date"] = pd.to_datetime(dax["Date"], format="%m/%d/%Y").dt.date
dax["Price"] = dax["Price"].astype(str).str.replace(",", "").astype(float)
dax = dax[["Date", "Price"]].sort_values("Date").reset_index(drop=True)
dax = dax.rename(columns={"Price": "DAX_Price"})

# --- Trading calendar: built from the DAX file, since it already contains
# only real trading days (weekends/holidays are absent) ---
trading_days = pd.Series(sorted(dax["Date"].unique()))

def next_trading_day(after_date, n, calendar=trading_days):
    """
    nth trading day STRICTLY AFTER after_date (0 = first, 1 = second, 2 = third).
    Returns None if the dataset runs out of trading days (e.g. near the end).
    """
    later = calendar[calendar > after_date]
    if len(later) <= n:
        return None
    return later.iloc[n]

def last_trading_day_on_or_before(date, calendar=trading_days):
    """Anchor date: last real trading day at or before the match date."""
    earlier = calendar[calendar <= date]
    if len(earlier) == 0:
        return None
    return earlier.iloc[-1]

# --- Merge Dortmund + DAX prices onto shared trading dates ---
prices = pd.merge(stock_dortmund, dax, on="Date", how="inner")
price_lookup = prices.set_index("Date")

def get_price(date, col):
    if date is None or date not in price_lookup.index:
        return np.nan
    return price_lookup.loc[date, col]

# --- Build one row per match: anchor price + day0/day1/day2, both series ---
records = []
for _, row in match_dortmund.iterrows():
    match_date = row['Date']
    anchor_date = last_trading_day_on_or_before(match_date)
    if anchor_date is None:
        continue

    day0_date = next_trading_day(anchor_date, 0)
    day1_date = next_trading_day(anchor_date, 1)
    day2_date = next_trading_day(anchor_date, 2)
    if day2_date is None:
        continue

    records.append({
        'match_date': match_date, 'result': row['Result'],
        'anchor_price': get_price(anchor_date, 'Price'),
        'day0_price':   get_price(day0_date, 'Price'),
        'day1_price':   get_price(day1_date, 'Price'),
        'day2_price':   get_price(day2_date, 'Price'),
        'anchor_dax':   get_price(anchor_date, 'DAX_Price'),
        'day0_dax':     get_price(day0_date, 'DAX_Price'),
        'day1_dax':     get_price(day1_date, 'DAX_Price'),
        'day2_dax':     get_price(day2_date, 'DAX_Price'),
    })

events = pd.DataFrame(records).dropna().reset_index(drop=True)
print(f"Built {len(events)} matches with complete event windows (out of {len(match_dortmund)} total matches).")

# --- Return calculators ---
def simple_return(p_start, p_end):
    return (p_end - p_start) / p_start

def log_return(p_start, p_end):
    return np.log(p_end / p_start)

def abnormal_return(stock_start, stock_end, market_start, market_end):
    """Market-adjusted abnormal return: stock's simple return minus the
    market's (DAX) simple return over the same window."""
    return simple_return(stock_start, stock_end) - simple_return(market_start, market_end)

WINDOWS = {
    'Day 0':     ('day0_price', 'day0_dax'),
    'Day 0 + 1': ('day1_price', 'day1_dax'),
    'Day 0 + 2': ('day2_price', 'day2_dax'),
}
RETURN_TYPES = ['Simple Return', 'Log Return', 'Abnormal Return']

# --- Compute all 9 combinations ---
results_summary = []
per_match_returns = {}

for window_name, (price_col, dax_col) in WINDOWS.items():
    simple_vals   = simple_return(events['anchor_price'], events[price_col])
    log_vals      = log_return(events['anchor_price'], events[price_col])
    abnormal_vals = abnormal_return(events['anchor_price'], events[price_col],
                                     events['anchor_dax'], events[dax_col])

    for return_name, vals in zip(RETURN_TYPES, [simple_vals, log_vals, abnormal_vals]):
        vals = vals.dropna()
        t_stat, p_val = stats.ttest_1samp(vals, popmean=0)
        results_summary.append({
            'Window': window_name, 'Return Type': return_name, 'N Matches': len(vals),
            'Mean Return': vals.mean(), 'Std Dev': vals.std(),
            't-stat': t_stat, 'p-value': p_val,
        })
        per_match_returns[(window_name, return_name)] = vals

        print("="*78)
        print(f"{return_name.upper()} — {window_name.upper()}")
        print("="*78)
        print(f"N matches:     {len(vals)}")
        print(f"Mean return:   {vals.mean():.5f}")
        print(f"Std dev:       {vals.std():.5f}")
        print(f"t-statistic:   {t_stat:.3f}")
        sig = "(significant at 5%)" if p_val < 0.05 else "(not significant at 5%)"
        print(f"p-value:       {p_val:.4f} {sig}")
        print()

# --- Full 9-row summary table ---
summary_df = pd.DataFrame(results_summary)
print("="*78)
print("FULL EVENT STUDY SUMMARY — ALL 9 COMBINATIONS")
print("="*78)
print(summary_df.round(5).to_string(index=False))

# --- Bonus: breakdown by match result (Win / Draw / Loss), log return shown as example ---
print("\n" + "="*78)
print("BREAKDOWN BY MATCH RESULT (Log Return shown as example — same idea applies to all 9)")
print("="*78)
for window_name, (price_col, dax_col) in WINDOWS.items():
    events['tmp_log_return'] = log_return(events['anchor_price'], events[price_col])
    by_result = events.groupby('result')['tmp_log_return'].agg(['count', 'mean', 'std'])
    print(f"\n{window_name}:")
    print(by_result.round(5).to_string())
events = events.drop(columns=['tmp_log_return'])

print("\nALL PARTS COMPLETED WITHOUT ERROR")